In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv(r"E:\engg\Data\movies_data\IMDb Movies India.csv", encoding="latin1")
df.head()

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,,NaN,NaN,Drama,NaN,NaN,J.S. Randhawa,Manmauji,Birbal,Rajendra Bhatia
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
2,#Homecoming,(2021),90 min,"Drama, Musical",NaN,NaN,Soumyajit Majumdar,Sayani Gupta,Plabita Borthakur,Roy Angana
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
4,...And Once Again,(2010),105 min,Drama,NaN,NaN,Amol Palekar,Rajat Kapoor,Rituparna Sengupta,Antara Mali


In [3]:
df_clean = df.copy()

In [4]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15509 entries, 0 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      15509 non-null  object 
 1   Year      14981 non-null  object 
 2   Duration  7240 non-null   object 
 3   Genre     13632 non-null  object 
 4   Rating    7919 non-null   float64
 5   Votes     7920 non-null   object 
 6   Director  14984 non-null  object 
 7   Actor 1   13892 non-null  object 
 8   Actor 2   13125 non-null  object 
 9   Actor 3   12365 non-null  object 
dtypes: float64(1), object(9)
memory usage: 1.2+ MB


In [5]:
df_clean.columns

Index(['Name', 'Year', 'Duration', 'Genre', 'Rating', 'Votes', 'Director',
       'Actor 1', 'Actor 2', 'Actor 3'],
      dtype='object')

In [6]:
df_clean.isnull().sum()

Name           0
Year         528
Duration    8269
Genre       1877
Rating      7590
Votes       7589
Director     525
Actor 1     1617
Actor 2     2384
Actor 3     3144
dtype: int64

In [7]:
df_clean.duplicated().sum()

np.int64(6)

In [8]:
df_clean.drop_duplicates(inplace=True)

In [9]:
df_clean.drop(columns=['Name'], inplace=True)

In [10]:
df_clean["Year"]=df_clean["Year"].str.replace("(", "")
df_clean["Year"]=df_clean["Year"].str.replace(")", "")

In [11]:
df_clean['Rating'] = pd.to_numeric(df_clean['Rating'], errors='coerce')

In [12]:
df_clean = df_clean.dropna(subset=['Rating'])

In [13]:
df_clean['Duration'] = df_clean['Duration'].str.replace(' min', '', regex=False).astype(float)

In [14]:
df_clean['Votes'] = df_clean['Votes'].astype(str).str.replace(',', '', regex=False)
df_clean['Votes'] = pd.to_numeric(df_clean['Votes'], errors='coerce').astype('Int64')

In [15]:
df_clean = df_clean.dropna(subset=['Rating']).reset_index(drop=True)

In [16]:
df_clean

,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,2019,109.0,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
1,2019,110.0,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
2,1997,147.0,"Comedy, Drama, Musical",4.7,827,Rahul Rawail,Bobby Deol,Aishwarya Rai Bachchan,Shammi Kapoor
3,2005,142.0,"Drama, Romance, War",7.4,1086,Shoojit Sircar,Jimmy Sheirgill,Minissha Lamba,Yashpal Sharma
4,2012,82.0,"Horror, Mystery, Thriller",5.6,326,Allyson Patel,Yash Dave,Muntazir Ahmad,Kiran Bhatia
...,...,...,...,...,...,...,...,...,...
7914,1992,NaN,"Action, Crime, Drama",5.3,135,Bharat Rangachary,Dharmendra,Moushumi Chatterjee,Govinda
7915,1989,125.0,"Action, Crime, Drama",5.8,44,S.P. Muthuraman,Chiranjeevi,Jayamalini,Rajinikanth
7916,1988,NaN,Action,4.6,11,Mahendra Shah,Naseeruddin Shah,Sumeet Saigal,Suparna Anand
7917,1999,129.0,"Action, Drama",4.5,655,Kuku Kohli,Akshay Kumar,Twinkle Khanna,Aruna Irani


In [17]:
df_clean['Genre'] = df_clean['Genre'].fillna('Unknown')
for col in ['Director', 'Actor 1', 'Actor 2', 'Actor 3']:
    df_clean[col] = df_clean[col].fillna('Unknown')

In [18]:
genre_dummies = df_clean['Genre'].str.get_dummies(sep=', ')
df_clean = pd.concat([df_clean, genre_dummies], axis=1)

In [19]:
import pickle, os

os.makedirs('model_artifacts', exist_ok=True)

genre_columns = genre_dummies.columns.tolist()
with open('genre.pkl', 'wb') as f:
    pickle.dump(genre_columns, f)


In [20]:
for col in ['Director', 'Actor 1', 'Actor 2', 'Actor 3']:
    freq = df_clean[col].value_counts()
    df_clean[col + '_freq'] = df_clean[col].map(freq)

In [21]:
freq_maps = {}
for col in ['Director', 'Actor 1', 'Actor 2', 'Actor 3']:
    freq_maps[col] = df_clean[col].value_counts().to_dict()

with open('actor_freq_mapping.pkl', 'wb') as f:
    pickle.dump(freq_maps, f)

print("Saved frequency maps for:", list(freq_maps.keys()))

Saved frequency maps for: ['Director', 'Actor 1', 'Actor 2', 'Actor 3']


In [22]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="mean")
df_clean['Duration']=imputer.fit_transform(df_clean[['Duration']])

In [23]:
with open('duration_imputer.pkl', 'wb') as f:
    pickle.dump(imputer, f)

In [24]:
df_clean = df_clean.drop(columns=['Genre', 'Director', 'Actor 1', 'Actor 2', 'Actor 3'])

In [25]:
df_clean["Duration"] = df_clean["Duration"].round(1)

In [26]:
df_clean

,Year,Duration,Rating,Votes,Action,Adventure,Animation,Biography,Comedy,Crime,...,Sci-Fi,Sport,Thriller,Unknown,War,Western,Director_freq,Actor 1_freq,Actor 2_freq,Actor 3_freq
0,2019,109.0,7.0,8,0,0,0,0,0,0,...,0,0,0,0,0,0,1,2,1,1
1,2019,110.0,4.4,35,0,0,0,0,1,0,...,0,0,0,0,0,0,1,5,1,2
2,1997,147.0,4.7,827,0,0,0,0,1,0,...,0,0,0,0,0,0,17,18,15,13
3,2005,142.0,7.4,1086,0,0,0,0,0,0,...,0,0,0,0,1,0,7,25,4,8
4,2012,82.0,5.6,326,0,0,0,0,0,0,...,0,0,1,0,0,0,1,1,6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7914,1992,132.3,5.3,135,1,0,0,0,0,1,...,0,0,0,0,0,0,7,134,28,23
7915,1989,125.0,5.8,44,1,0,0,0,0,1,...,0,0,0,0,0,0,14,14,1,8
7916,1988,132.3,4.6,11,1,0,0,0,0,0,...,0,0,0,0,0,0,4,47,2,1
7917,1999,129.0,4.5,655,1,0,0,0,0,0,...,0,0,0,0,0,0,8,82,11,35


In [27]:
df_clean.to_csv("clean_movies.csv", index=False)

In [28]:
feature_columns = df_clean.drop(columns=['Rating']).columns.tolist()

with open('feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)


In [33]:
'''Note - Problem encountered- Decision regarding the actor columns. 
This problem was solved with the use of AI'''

'Note - Problem encountered- Decision regarding the actor columns. \nThis problem was solved with the use of AI'